# Response distributions of the literature panel, and the loss weighting they motivate

## Where this sits

Two decisions from 25–27.07.2026 need data behind them before anything is trained:

1. **Drugs come from the literature, not from a filter.** The eight compounds were selected from
   *published* cell-line sensitivity determinants, so the choice is defensible by citation and does not
   depend on our own labels ([Step 05](../../docs/steps/05-multitask-results.md)). This notebook shows what
   that buys on the data side.
2. **`auc_z` is retired as the target.** Its centering was inert (the per-drug head bias absorbs it) and
   its scaling — dividing by each drug's standard deviation — forced every compound to variance 1,
   including compounds whose spread is mostly assay noise
   ([Step 03](../../docs/steps/03-model-and-training-design.md)). The target becomes **raw `auc`**.

Retiring `auc_z` reopens the question it was introduced to solve, so this notebook answers it in two
parts — and they turn out to have different answers.

## The two questions

**Between drugs — is a per-drug weight still needed?** The June defect was that squared error scales
with a drug's variance, so a minority of wide-spread compounds captured the shared trunk and drove every
correlation to zero. Section 2 measures whether that imbalance still exists on eight comparable
compounds. **Answer: no.**

**Within a drug — is a per-sample weight needed?** The cell lines are not spread evenly over a drug's
response range: most sit in a narrow band and the pharmacologically interesting extremes are sparse.
Under unweighted squared error the crowded middle owns the gradient and the optimum shrinks toward the
drug mean — which is precisely the documented behaviour of the current model (`pred_std` 0.53 for PCA and
0.47 for scGPT against a true spread of 1.0). **Answer: yes**, and sections 3–5 build it.

This second case is the regression analogue of class imbalance. The established remedy is to weight each
observation by the inverse of a *smoothed* estimate of the label density:

> Yang, Zha, Chen, Wang & Katabi. *Delving into Deep Imbalanced Regression.* ICML 2021 — Label
> Distribution Smoothing.  
> Steininger, Kobs, Davidson, Krause & Hotho. *Density-based weighting for imbalanced regression.*
> Machine Learning 110, 2021 — DenseWeight, which adds the exponent used below.

## What is decided here, and why each choice was made

| Choice | Value | Reason |
|---|---|---|
| target | raw `auc` | interpretable units; an MSE of 0.0025 is 5 viability points |
| per-drug weight | none | section 2 shows the imbalance is 2.5x, not 81x |
| density estimate | Gaussian KDE, Scott's rule | continuous equivalent of LDS's binned-and-smoothed density; bandwidth is the one real knob |
| density scope | per drug | the metric is *within*-drug rank correlation, so the weighting must target that and not potency differences between drugs |
| exponent `ALPHA` | 0.5 | full inverse density (1.0) saturates the cap over wide ranges — see section 4 |
| cap | 3x | with ~170 lines per drug a looser cap lets a handful of lines dominate |
| winsorize at 1.1 | yes | see section 3: the sparsest region is `auc > 1.1`, which is assay artifact, and inverse-density weighting would otherwise give the least trustworthy points the most influence |

The last three were **not** the first choice — they are corrections made after plotting the first
version, which is recorded in section 4.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, skew

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.preprocessing.layout import PipelinePaths
from scripts.training.density_weighting import (
    DEFAULT_ALPHA,
    DEFAULT_CAP,
    fit_weight_fn,
)

OUT = ROOT / 'notebooks' / 'outputs' / 'panel'
OUT.mkdir(parents=True, exist_ok=True)

# ---- every knob of the weighting scheme, in one place -------------------------
SCORE, VARIANT = 'auc_cc', 'hvg5000'  # CurveCurator AUC: no centering, no scaling
WINSOR = 1.1   # RETIRED 11.08.2026 -- kept local so section 3 still runs as a record of
               # the design that was rejected. The pipeline no longer clips anything:
               # docs/steps/01-datasets-and-harmonization.md, the target section.
ALPHA = DEFAULT_ALPHA              # w ~ density^(-ALPHA); 1.0 = full inverse (section 4)
CAP = DEFAULT_CAP                  # max / min weight factor after normalizing to mean 1

# One accent plus neutrals; two colours only, so no colour-vision separation problem.
ACCENT, CONTEXT, INK, MUTED = '#2a78d6', '#c9c9c4', '#0b0b0b', '#52514e'
plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'text.color': INK,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.titlesize': 9,
    'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'grid.color': '#e8e8e4', 'grid.linewidth': 0.6, 'font.size': 8,
})

paths = PipelinePaths.build(None, VARIANT, SCORE)
a = ad.read_h5ad(paths.targets_h5ad, backed='r')  # backed: .X stays on disk, we touch only obsm/uns
print(paths.targets_h5ad.name, '| score =', a.uns['ctrp_score'])

## 1. Collapse to the real (cell line x drug) measurements

The label is bulk: CTRPv2 reports **one value per cell line**, which `ctrp_to_h5ad.py` broadcasts to
every cell of that line. Every per-drug statistic must therefore be computed on the **line-level**
matrix. Computing it per cell would count cell-rich lines many times over — cells per line range from
**56 to 1,990** in this dataset, a factor of 35 that reflects sequencing depth, not biology.

In [ ]:
Y = np.asarray(a.obsm['Y_ctrp'], dtype=float)  # raw auc, broadcast to cells
M = np.asarray(a.obsm['M_ctrp'], dtype=bool)   # True where the line was screened on that drug
drugs = list(a.uns['ctrp_drugs'])
lines = a.obs['Cell_line'].astype(str).to_numpy()
uniq = np.unique(lines)

A = np.full((len(uniq), len(drugs)), np.nan)   # (cell line x drug) raw AUC
for i, ln in enumerate(uniq):
    idx = np.flatnonzero(lines == ln)
    m = M[idx]
    has, cnt = m.any(0), m.sum(0)
    s = np.where(m, Y[idx], 0.0).sum(0)
    A[i, has] = s[has] / cnt[has]              # all a line's cells carry the same value

A = pd.DataFrame(A, index=uniq, columns=drugs)
print(f'{A.shape[0]} cell lines x {A.shape[1]} drugs | labelled lines: {int(A.notna().any(axis=1).sum())}')

## 2. The panel, and why it needs no per-drug weighting

The eight compounds and the publication behind each are tabulated in
[Step 05](../../docs/steps/05-multitask-results.md). Briefly: `methotrexate` (SLC19A1 transport),
`dasatinib` (six-gene expression signature), `paclitaxel` and `vincristine` (ABCB1 efflux / TUBB3),
`afatinib` (EGFR+ERBB2 amplification), `topotecan` (SLFN11), `tanespimycin` (NQO1 bioactivation),
`selumetinib` (BRAF/RAS).

**The point of this section.** A drug's share of an unweighted squared-error loss scales with its
variance. Across all 545 CTRPv2 compounds those variances differ by orders of magnitude, which is what
let a minority of wide compounds capture the shared trunk — the defect `auc_z` was introduced to fix. If
the panel's spreads are comparable, the defect cannot arise and no correction is required. That matters
because the correction has a side effect of its own: dividing by sigma rescales a noise-dominated drug up
to full weight, which is why `auc_z` was the wrong instrument rather than merely an imperfect one.

### How the eight were selected — derived here, not asserted

The panel used to be a hard-coded list, with the funnel counts living only in a shell command. The
selection is reconstructed below so the numbers are checkable and the citations travel with the data.

**What can and cannot be automated.** The *pool* is a filter and is derived here: CTRPv2 compounds ->
single agents -> approved or in clinical trials. The *choice within that pool* is a literature judgement
and cannot be; it is carried as a table of determinants with references, entered by hand. The code's job
is to prove the eight lie inside the derived pool and to keep the evidence attached to them.

**And the part that is not independent of our data.** Candidates were ranked by `min(kill, spare)` on our
own AUCs *before* the literature criterion was applied, so compounds with a published determinant but
little spread here dropped out. That step is reproduced below rather than only confessed in prose — the
drugs it discarded are printed, so the non-independence is visible.


In [ ]:
CATALOG = ROOT / 'data' / 'drug' / 'all_sources_drug_catalog.csv'

# The literature layer: one published determinant per compound, with its reference and -- decisive for
# the panel's own prediction -- whether that determinant is visible in expression or only in genomics.
DETERMINANTS = {
    'methotrexate':  ('SLC19A1 (reduced folate carrier) governs uptake; its loss is the classical '
                      'resistance mechanism', 'Zhao & Goldman 2014; Wright et al., Nature 2022', 'expression'),
    'dasatinib':     ('six-gene expression signature, 92% / 83% accuracy in held-out breast / lung lines; LYN',
                      'Huang et al., Cancer Res 2007; Oncotarget 2016', 'expression'),
    'paclitaxel':    ('ABCB1 efflux and TUBB3', 'Oncotarget 2016; Br J Cancer 2016', 'expression'),
    'vincristine':   ('same ABCB1 / TUBB3 axis (shared microtubule-disruptor resistance)',
                      'Oncotarget 2016; Br J Cancer 2016', 'expression'),
    'afatinib':      ('EGFR + ERBB2 co-amplification; receptor expression alone did NOT predict it',
                      'Cancer Discov 2019; Br J Cancer 2011', 'mutation'),
    'topotecan':     ('SLFN11 expression, the canonical topoisomerase-inhibitor marker',
                      'Zoppoli et al., PNAS 2012; PLOS One 2019', 'expression'),
    'tanespimycin':  ('NQO1 bioactivates the benzoquinone; confirmed in CCLE and GDSC, 7 cancer types',
                      'PLOS One 2016; Br J Cancer 2014', 'expression'),
    'selumetinib':   ('BRAF / RAS mutation', 'Mol Cancer Ther 2010', 'mutation'),
}

cat = pd.read_csv(CATALOG)
cat = cat[cat.dataset == 'CTRPv2'].copy()
cat['name'] = cat.compound_name_norm.str.strip().str.lower()
cat = cat.drop_duplicates('name').set_index('name')

combo = (cat.screen == 'combination') | cat.index.str.contains(':')
clinical = cat.compound_status.isin(['clinical', 'FDA'])
pool = cat[~combo & clinical]

print('selection funnel')
print(f'  CTRPv2 compounds                     {len(cat):4d}')
print(f'  single agents (drop combinations)    {int((~combo).sum()):4d}')
print(f'  approved or in clinical trials       {len(pool):4d}   <- the pool the panel is chosen from')
print(f'  with a published determinant          {len(DETERMINANTS):4d}   <- the panel')

missing = [d for d in DETERMINANTS if d not in pool.index]
assert not missing, f'panel drugs outside the derived pool: {missing}'
print('\nall eight lie inside the derived pool  ✓')

In [ ]:
# The label-dependent step, made visible: candidates were ranked by min(kill, spare) on our own AUCs
# before the literature criterion was applied. Compounds with a published determinant but little spread
# here were therefore never considered -- which is why the panel is spread-verified, not label-blind.
pool_stats = pd.DataFrame({
    'kills': (A[pool.index.intersection(A.columns)] <= 0.5).sum(),
    'spares': (A[pool.index.intersection(A.columns)] >= 0.8).sum(),
})
pool_stats['balance'] = pool_stats.min(axis=1)
dropped = ['sirolimus', 'neratinib', 'clofarabine', 'cytarabine hydrochloride', 'gdc-0941']
print('known determinants that fell out on OUR spread, not on the literature:')
print(pool_stats.reindex([d for d in dropped if d in pool_stats.index]).to_string())
print(f'\nfor comparison, the panel: balance '
      f'{pool_stats.reindex(list(DETERMINANTS)).balance.min():.0f}-'
      f'{pool_stats.reindex(list(DETERMINANTS)).balance.max():.0f}')

sel = pd.DataFrame([
    {'drug': d, 'target': pool.loc[d, 'target'], 'moa': pool.loc[d, 'moa_or_pathway'],
     'status': pool.loc[d, 'compound_status'], 'determinant': det, 'reference': ref,
     'determinant_type': kind, 'n_lines': int(A[d].notna().sum()),
     'auc_std': float(A[d].std(ddof=0)), 'kills': int((A[d] <= 0.5).sum()),
     'spares': int((A[d] >= 0.8).sum())}
    for d, (det, ref, kind) in DETERMINANTS.items()]).set_index('drug')
sel.to_csv(OUT / 'panel_selection.csv')
print()
print(sel[['status', 'determinant_type', 'n_lines', 'auc_std', 'kills', 'spares']].to_string())

PANEL = list(DETERMINANTS)   # <- the panel is now derived from the evidence table, not typed twice

In [ ]:
missing = [d for d in PANEL if d not in A.columns]
assert not missing, f'not in CTRPv2: {missing}'

sd_all = A.std(axis=0, ddof=0)
sd_panel = sd_all[PANEL]
print(f'all {len(sd_all)} drugs : sigma {sd_all.min():.3f} .. {sd_all.max():.3f} '
      f'-> loss-share ratio {(sd_all.max() / sd_all.min())**2:.0f}x')
print(f'the {len(PANEL)} panel   : sigma {sd_panel.min():.3f} .. {sd_panel.max():.3f} '
      f'-> loss-share ratio {(sd_panel.max() / sd_panel.min())**2:.1f}x')


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 2.5))
rng = np.random.default_rng(42)  # vertical jitter is cosmetic only; seeded for reproducibility
ax.scatter(sd_all.values, rng.uniform(-0.35, 0.35, len(sd_all)), s=9,
           color=CONTEXT, edgecolors='none', label=f'all {len(sd_all)} CTRPv2 drugs')
ax.scatter(sd_panel.values, np.zeros(len(sd_panel)), s=46, color=ACCENT,
           edgecolors='white', linewidths=1.0, zorder=3, label=f'literature panel ({len(PANEL)})')
for d in PANEL:
    ax.annotate(d, (sd_panel[d], 0), textcoords='offset points', xytext=(0, 11),
                ha='center', fontsize=6.5, rotation=45, color=INK)
ax.set_yticks([])
ax.set_ylim(-0.75, 0.95)
ax.set_xlabel('per-drug spread of AUC across cell lines  (sigma, raw scale)')
ax.set_title('The panel sits in a narrow, high-spread band — so no per-drug loss weighting is needed',
             loc='left', color=INK)
ax.grid(axis='x')
ax.set_axisbelow(True)
ax.legend(frameon=False, loc='lower right', fontsize=7)
fig.savefig(OUT / 'panel_sigma_context.png')
plt.show()

## 3. Winsorizing at AUC = 1.1, and why it has to happen before weighting

`auc = 1.0` means the treated cells matched the untreated control. Values **above** 1.0 say the compound
made the cells grow *better* than control. A little of that is expected from assay scatter around 1.0;
values well beyond it are not a pharmacological signal that the model should be asked to rank — they are
the noisy end of a viability readout.

This interacts badly with inverse-density weighting, and it is the reason this section exists. The
far-upper tail is the **sparsest** region of every drug's distribution, so inverse density assigns it the
**highest** weight. Left alone, the scheme would hand maximum influence to the least trustworthy
measurements — the opposite of what it is for. Clipping to `WINSOR = 1.1` keeps those cell lines in the
training set at a defensible value ('no effect') instead of deleting them, while removing their ability
to dominate through the weight.

In [ ]:
raw_vals = {d: A[d].dropna().to_numpy() for d in PANEL}
vals = {d: np.clip(v, None, WINSOR) for d, v in raw_vals.items()}

clip_report = pd.DataFrame({
    'n_lines': {d: len(v) for d, v in raw_vals.items()},
    'max_raw_auc': {d: v.max() for d, v in raw_vals.items()},
    'n_clipped': {d: int((v > WINSOR).sum()) for d, v in raw_vals.items()},
    'pct_clipped': {d: 100 * (v > WINSOR).mean() for d, v in raw_vals.items()},
}).round(2)
tot_clipped, tot_lines = int(clip_report['n_clipped'].sum()), int(clip_report['n_lines'].sum())
print(f'winsorized {tot_clipped} of {tot_lines} measurements '
      f'({100 * tot_clipped / tot_lines:.2f} %) at auc > {WINSOR}')
clip_report

### The distributions themselves

One panel per drug: how the screened cell lines spread over the response range, after winsorizing. This
is the quantity the model has to rank, and its shape is what the weighting responds to.

Read them for two things — where the mass sits (the region an unweighted loss will optimize for) and how
thin the low-AUC tail is (the sensitive lines, which are the pharmacologically interesting ones).

In [ ]:
XMAX = WINSOR + 0.05
NBINS = 22
fig, axes = plt.subplots(2, 4, figsize=(11, 4.6), sharex=True)
for ax, d in zip(axes.ravel(), PANEL):
    v = vals[d]
    ax.hist(v, bins=NBINS, range=(0, XMAX), color=ACCENT, alpha=0.5,
            edgecolor='white', linewidth=0.5)
    grid = np.linspace(0, XMAX, 300)
    ax.plot(grid, gaussian_kde(v)(grid) * len(v) * (XMAX / NBINS), color=ACCENT, linewidth=1.6)
    ax.axvline(v.mean(), color=INK, linewidth=1.0, linestyle='--')
    ax.set_title(f'{d}   n={len(v)}  sigma={v.std(ddof=0):.3f}  skew={skew(v):+.2f}', loc='left')
    ax.grid(axis='y')
    ax.set_axisbelow(True)
for ax in axes[1]:
    ax.set_xlabel('AUC  (low = line is killed)')
for ax in axes[:, 0]:
    ax.set_ylabel('cell lines')
fig.suptitle('Response distribution per drug across cell lines, winsorized at '
             f'{WINSOR} — dashed line = drug mean', x=0.005, ha='left', fontsize=10, color=INK)
fig.tight_layout()
fig.savefig(OUT / 'panel_distributions.png')
plt.show()

## 4. The weighting scheme, and how its parameters were chosen

For each drug independently, over that drug's cell-line values:

```
p     = gaussian_kde(v)          smoothed density of the drug's values
w_raw = p(v) ** (-ALPHA)         rarer value -> larger weight
w     = w_raw / mean(w_raw)      normalize so the drug's total contribution is unchanged
w     = clip(w, 1/CAP, CAP)      then renormalize; iterate until stable
```

Normalizing per drug to mean 1 means this acts **strictly within** a drug and cannot reintroduce the
between-drug imbalance of section 2.

**Why `ALPHA = 0.5` and `CAP = 3` rather than the textbook `ALPHA = 1`.** The first version of this
notebook used the full inverse density (`ALPHA = 1`) with `CAP = 10`. Plotting it showed the scheme was
not doing what it was supposed to: the weight curve **saturated at the cap over wide stretches** of the
response range, so the cap — an arbitrary safety limit — was setting the weights rather than the density.
With roughly 170 lines per drug, a handful of extreme lines would then have driven the fit. The exponent
is DenseWeight's remedy for exactly this: it compresses the weight range so the density still orders the
samples but no longer dictates the magnitude.

*(That first version also had a bug worth recording: clipping and then renormalizing pushes values back
above the cap, so the reported maximum was 13.1 under a cap of 10. The loop below iterates to a fixed
point and asserts the result.)*

The figure below is the justification for the exponent — it shows what the three candidate values
actually do to one drug.

In [ ]:
# The weighting itself lives in scripts/training/density_weighting.py, so this notebook and the
# training run in 14 cannot drift apart. A WeightFn is callable on any values -- the fitting values,
# held-out cells, or a grid for plotting -- with the normalization frozen at fit time.
#
# Why ALPHA = 0.5: what the candidates do to one drug's weight curve.
demo = 'afatinib'  # largest tail-vs-middle ratio in the panel, so the effect is clearest
v_demo = vals[demo]
grid_demo = np.linspace(v_demo.min(), v_demo.max(), 300)

fig, ax = plt.subplots(figsize=(6.4, 2.8))
for alpha, style in [(0.25, ':'), (0.5, '-'), (1.0, '--')]:
    fn = fit_weight_fn(v_demo, alpha=alpha, cap=CAP)
    ax.plot(grid_demo, fn(grid_demo), style, color=ACCENT, linewidth=1.6,
            label=f'alpha = {alpha}' + ('  (chosen)' if alpha == ALPHA else ''))
ax.axhline(1.0, color=MUTED, linewidth=0.8, linestyle=':')
ax.axhline(CAP, color=MUTED, linewidth=0.8)
ax.annotate(f'cap = {CAP:g}', (v_demo.min(), CAP), textcoords='offset points', xytext=(2, 3),
            fontsize=7, color=MUTED)
ax.set_xlabel(f'AUC  ({demo})')
ax.set_ylabel('sample weight')
ax.set_title('alpha = 1 pins the weight to the cap across the tail; 0.5 keeps the density in charge',
             loc='left', color=INK)
ax.grid(axis='y')
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=7)
fig.savefig(OUT / 'panel_alpha_choice.png')
plt.show()


## 5. The weights the training run will use

Each dot is one cell line at its weight; the curve is the weight function over the response range. A line
sitting where many lines sit counts less than one, a line in a sparse region counts more, and the ratio
between the two is bounded by the cap.

In the summary table, `w_sensitive_decile` is the mean weight of the 10 % most-killed lines — the ones
the model is meant to find — against `w_mid_50pct` for the crowded middle. Their ratio is the size of the
intervention.

In [ ]:
weights, rows = {}, []
for d in PANEL:
    v = vals[d]
    fn = fit_weight_fn(v, alpha=ALPHA, cap=CAP)   # shared implementation, see cell above
    w = fn(v)
    weights[d] = (v, w, fn)
    lo = np.quantile(v, 0.1)
    q25, q75 = np.quantile(v, [0.25, 0.75])
    rows.append({
        'drug': d, 'n_lines': len(v), 'mean': v.mean(), 'sigma': v.std(ddof=0), 'skew': skew(v),
        'w_min': w.min(), 'w_max': w.max(),
        'w_sensitive_decile': w[v <= lo].mean(),
        'w_mid_50pct': w[(v > q25) & (v < q75)].mean(),
    })

summary = pd.DataFrame(rows).set_index('drug')
summary['tail_vs_mid'] = summary['w_sensitive_decile'] / summary['w_mid_50pct']
summary = summary.round(3)
summary.to_csv(OUT / 'panel_weight_summary.csv')
print(f'alpha={ALPHA}  cap={CAP}  winsor={WINSOR}')
summary


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(11, 4.6), sharex=True, sharey=True)
for ax, d in zip(axes.ravel(), PANEL):
    v, w, fn = weights[d]
    grid = np.linspace(v.min(), v.max(), 300)
    ax.plot(grid, fn(grid), color=ACCENT, linewidth=1.6)
    ax.scatter(v, w, s=7, color=CONTEXT, edgecolors='none', zorder=2)
    ax.axhline(1.0, color=MUTED, linewidth=0.8, linestyle=':')
    ax.set_title(f'{d}   w {w.min():.2f}-{w.max():.2f}', loc='left')
    ax.grid(axis='y')
    ax.set_axisbelow(True)
for ax in axes[1]:
    ax.set_xlabel('AUC')
for ax in axes[:, 0]:
    ax.set_ylabel('sample weight')
fig.suptitle('Inverse-density weight per cell line — dotted line = unweighted (w = 1); dots are the lines',
             x=0.005, ha='left', fontsize=10, color=INK)
fig.tight_layout()
fig.savefig(OUT / 'panel_weights.png')
plt.show()

## 6. What this establishes, and what it does not

**Established, from the numbers above.**

- *Between drugs:* the panel's variances differ by 2.5x against 81x across all 545 compounds. An
  unweighted masked MSE on raw AUC cannot let one compound capture the shared trunk here, so no per-drug
  weight is needed and the `auc_z` standardization is not merely retired but unnecessary.
- *Within a drug:* the crowded middle carries a weight around 0.7-0.85 and the most-sensitive decile
  1.14-1.84, a ratio of 1.4x to 2.7x depending on the compound. Bounded, and acting strictly within each
  drug.

**The caveat that matters most, and it emerged from these numbers.** Before winsorizing, several drugs
looked strongly right-skewed (`topotecan` +2.42, `paclitaxel` +1.13) — the classic imbalanced-regression
picture. After clipping the 15 measurements above `auc` 1.1, every drug has |skew| <= 0.47 and several
are slightly *left*-skewed. **The long tail was almost entirely assay artifact, not biology.**

The consequence is that this is *not* a pathological distribution being repaired. These are close to
symmetric, unimodal distributions, and inverse-density weighting on them is a **deliberate re-emphasis of
the sparse extremes** — the strongly killed and strongly resistant lines, which are the ones that carry
the pharmacological question. That is defensible, but it has to be argued as a modelling choice about
what the model should be good at, not as a correction of broken data. Expect a modest effect.

**Further caveats.**

- The KDE bandwidth (Scott's rule) is an unexamined choice. A sensitivity check over bandwidth belongs
  here if the weighting turns out to matter.
- Winsorizing at 1.1 is a judgement about where assay artifact begins, not a measurement. It touches
  1.09 % of measurements but, as above, changes the shape statistics substantially.

**Expected consequence when this is trained — stated before running, not after.** MSE will get *worse*
and rank correlation *better*. Shrinking toward the mean is what a squared-error optimum requires
(`notebooks/10_diagnosis.ipynb`), so deliberately over-weighting sparse regions trades calibration for
ranking. On this target the MSE is close to its null model anyway, so the rank correlation is the number
that carries information. If both move in the same direction, something is wrong.

**Next**, per the agreed order in [TODO](../../docs/TODO.md): train the existing per-cell MLP on raw AUC with
these weights, everything else unchanged, so the effect is attributable. Only then MIL, against this as
the fixed target, with the per-cell MLP and ridge-on-line-means as the controls it has to beat.
